In [ ]:
#Montage du Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import kagglehub

# Emplacements du dépôt et des sauvegardes
repo_dir = "/content/drive/MyDrive/aasist"
SAVE_DIR = f"{repo_dir}/checkpoints"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "best_aasist_model.pth")
os.makedirs(SAVE_DIR, exist_ok=True)

if os.path.exists(repo_dir):
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.append(repo_dir)
    print(f"Dossier actif : {os.getcwd()}")
else:
    print(f"ERREUR : {repo_dir} est introuvable sur ton Google Drive.")

#Vérification GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil de calcul : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
if torch.cuda.is_available():
    print(f"Nom du GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU totale : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} Go")

#Import des modules spécifiques d'AASIST
from models.AASIST import Model as AASISTModel
import evaluation as eval_metric

print("Étape 1 : Modules AASIST et GPU prêts ")

Dossier actif : /content/drive/.shortcut-targets-by-id/1N1k8ETYX0C7LmiaF9FnLR3NYqk4yHoC-/aasist
Appareil de calcul : cuda (Tesla T4)
Nom du GPU : Tesla T4
Mémoire GPU totale : 15.64 Go
Étape 1 : Modules AASIST et GPU prêts 


In [ ]:
import pickle
#Téléchargement et localisation des données
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
base_dir = Path(path)

CACHE_FILE = f"{repo_dir}/index_cache.pkl"
flac_map = {}
train_txt, dev_txt, eval_txt = None, None, None

# 2. Vérification et chargement sécurisé du cache
use_cache = False
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, "rb") as f:
            cache_data = pickle.load(f)
            test_path = list(cache_data["flac_map"].values())[0]
            # Vérifier si le fichier existe toujours à cet emplacement Colab
            if os.path.exists(test_path):
                flac_map = cache_data["flac_map"]
                train_txt = cache_data["train_txt"]
                dev_txt = cache_data["dev_txt"]
                eval_txt = cache_data["eval_txt"]
                use_cache = True
                print(" Indexation chargée instantanément depuis le cache Google Drive !")
            else:
                print(" Cache détecté mais obsolète (changement de session Colab). Re-indexation...")
    except Exception as e:
        print(" Erreur de lecture du cache. Re-indexation...")

if not use_cache:
    print(" Indexation rapide des fichiers en cours (os.walk)...")
    for root, _, files in os.walk(base_dir):
        if "LA" in root:
            for f in files:
                full_path = Path(root) / f
                if f.endswith(".flac"):
                    flac_map[f[:-5]] = full_path
                elif f.endswith("train.trn.txt"):
                    train_txt = full_path
                elif f.endswith("dev.trl.txt"):
                    dev_txt = full_path
                elif f.endswith("eval.trl.txt"):
                    eval_txt = full_path

    # Sauvegarde du cache valide
    cache_data = {
        "flac_map": flac_map,
        "train_txt": train_txt,
        "dev_txt": dev_txt,
        "eval_txt": eval_txt
    }
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(cache_data, f)
    print(f" Indexation sauvegardée sur Google Drive : {CACHE_FILE}")

# 3. Fonction de parsing des protocoles
def parse_protocol(protocol_file):
    file_paths = []
    labels = {}
    with open(protocol_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                key, label = parts[1], parts[4]
                if key in flac_map:
                    file_paths.append(flac_map[key])
                    labels[key] = 1 if label == "bonafide" else 0
    return file_paths, labels

# 4. Parsing des ensembles
paths_train, labels_train = parse_protocol(train_txt)
paths_dev, labels_dev = parse_protocol(dev_txt)
paths_eval, labels_eval = parse_protocol(eval_txt)

Using Colab cache for faster access to the 'asvpoof-2019-dataset' dataset.
 Indexation chargée instantanément depuis le cache Google Drive !


In [ ]:
# Fonction d'ajustement de la taille audio (fixée à 64 600 échantillons)
def prepare_audio(path):
    waveform, _ = torchaudio.load(str(path))
    audio = waveform.squeeze(0)

    if len(audio) < 64600:
        audio = audio.repeat((64600 // len(audio)) + 1)[:64600]
    else:
        audio = audio[:64600]

    return audio.float()

# 2. Classe PyTorch Dataset
class ASVDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        key = path.stem
        audio = prepare_audio(path)
        label = torch.tensor(self.labels[key], dtype=torch.long)
        return audio, label, key

print("Étape 3 terminée : Dataset et préparation audio")

Étape 3 terminée : Dataset et préparation audio


In [ ]:
# Chargement du fichier de configuration AASIST
config_path = f"{repo_dir}/config/AASIST.conf"

with open(config_path, "r") as f:
    config = json.load(f)

# 2. Initialisation du modèle et transfert sur le device (GPU)
model = AASISTModel(config["model_config"]).to(device)

print("--- Étape 4 terminée : Modèle AASIST initialisé avec succès ! ---")

--- Étape 4 terminée : Modèle AASIST initialisé avec succès ! ---


In [ ]:
# 1. Fonction d'entraînement pour une époque
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss, right_predicts, total_samples = 0.0, 0, 0

    for x_batch, y_batch, _ in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        _, outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        predicts = torch.argmax(outputs, dim=1)
        right_predicts += torch.sum(predicts == y_batch).item()
        total_samples += batch_size

    return right_predicts / total_samples, total_loss / total_samples

# 2. Fonction de prédiction sur le set de validation/évaluation
def eval_epoch(model, dev_loader, device):
    model.eval()
    tracker = {}

    with torch.no_grad():
        for x_batch, _, keys in dev_loader:
            x_batch = x_batch.to(device)
            _, outputs = model(x_batch)
            bonafide_scores = outputs[:, 1].cpu().numpy()
            for key, score in zip(keys, bonafide_scores):
                tracker[key] = score

    return tracker

# 3. Fonction de calcul de l'EER (Equal Error Rate)
def compute_eer(tracker, protocol_file):
    target_scores, nontarget_scores = [], []
    with open(protocol_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                key, label = parts[1], parts[4]
                if key in tracker:
                    if label == "bonafide":
                        target_scores.append(tracker[key])
                    else:
                        nontarget_scores.append(tracker[key])

    eer, _ = eval_metric.compute_eer(np.array(target_scores), np.array(nontarget_scores))
    return eer * 100

print("--- Étape 5 terminée : Fonctions d'entraînement et calcul EER prêtes ! ---")

--- Étape 5 terminée : Fonctions d'entraînement et calcul EER prêtes ! ---


In [ ]:
# 1. Configuration des DataLoaders & Hyperparamètres
BATCH_SIZE = 8
train_loader = DataLoader(ASVDataset(paths_train, labels_train), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(ASVDataset(paths_dev, labels_dev), batch_size=BATCH_SIZE, shuffle=False)

# 2. Perte (CrossEntropy avec poids) & Optimiseur Adam
weights = torch.FloatTensor([0.1, 0.9]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["optim_config"]["base_lr"],
    weight_decay=config["optim_config"]["weight_decay"]
)

NUM_EPOCHS = 5
BEST_EER = 100.0
SAVE_DIR = "/content/drive/MyDrive/aasist/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "best_aasist_model.pth")

# 3. Reprise automatique depuis le checkpoint de Google Drive
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print("Modèle précédent trouvé sur Drive ! Chargement et évaluation de l'EER initial...")
    initial_tracker = eval_epoch(model, dev_loader, device)
    BEST_EER = compute_eer(initial_tracker, dev_txt)
    print(f"Meilleur EER actuel (référence) : {BEST_EER:.2f}%")

# 4. Boucle principale d'entraînement
print("\n--- Début de l'entraînement sur GPU ---")
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nÉPOQUE [{epoch}/{NUM_EPOCHS}]")
    train_acc, train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"  [TRAIN] Loss : {train_loss:.4f} | Accuracy : {train_acc * 100:.2f}%")

    tracker = eval_epoch(model, dev_loader, device)
    dev_eer = compute_eer(tracker, dev_txt)
    print(f"  [EVAL] Dev EER : {dev_eer:.2f}%")

    # Sauvegarde si nouveau record
    if dev_eer < BEST_EER:
        print(f"  --> NOUVEAU RECORD ! EER : {BEST_EER:.2f}% -> {dev_eer:.2f}%")
        BEST_EER = dev_eer
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

print("\n--- Entraînement terminé ---")

Modèle précédent trouvé sur Drive ! Chargement et évaluation de l'EER initial...
Meilleur EER actuel (référence) : 1.45%

--- Début de l'entraînement sur GPU ---

ÉPOQUE [1/5]
  [TRAIN] Loss : 0.0981 | Accuracy : 97.47%
  [EVAL] Dev EER : 3.60%

ÉPOQUE [2/5]
  [TRAIN] Loss : 0.0723 | Accuracy : 98.08%
  [EVAL] Dev EER : 1.22%
  --> NOUVEAU RECORD ! EER : 1.45% -> 1.22%

ÉPOQUE [3/5]
  [TRAIN] Loss : 0.0627 | Accuracy : 98.19%
  [EVAL] Dev EER : 1.64%

ÉPOQUE [4/5]
  [TRAIN] Loss : 0.0633 | Accuracy : 98.35%
  [EVAL] Dev EER : 1.22%

ÉPOQUE [5/5]
  [TRAIN] Loss : 0.0528 | Accuracy : 98.73%
  [EVAL] Dev EER : 1.77%

--- Entraînement terminé ---


In [ ]:
# Cellule 7 : Évaluation finale sur le jeu d'évaluation (Eval set)
import os
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader

print("--- Chargement du meilleur modèle pour l'évaluation ---")

# 1. Chargement des poids sauvegardés
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Modèle chargé avec succès depuis : {MODEL_SAVE_PATH}")
else:
    print(f"Erreur : Aucun fichier trouvé à l'emplacement {MODEL_SAVE_PATH}")

# 2. Création du DataLoader d'évaluation
eval_loader = DataLoader(
    ASVDataset(paths_eval, labels_eval), batch_size=BATCH_SIZE, shuffle=False
)

# 3. Calcul des prédictions
print("Évaluation en cours sur le jeu Eval (71 237 audios)...")

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_data in eval_loader:
        # Récupération sécurisée du tensor audio et des étiquettes
        batch_x, batch_y = batch_data[0], batch_data[1]
        batch_x = batch_x.to(device)

        # Inférence AASIST
        _, outputs = model(batch_x)

        # Prédictions du modèle (Classe avec le score logit/softmax le plus élevé)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(batch_y.numpy())

# Conversion en tableaux NumPy
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# 4. Ajustement des classes pour Scikit-Learn
# Dans votre parse_protocol : 1 = BONAFIDE (Authentique), 0 = SPOOF (Falsifié)
# On définit pos_label=0 pour évaluer spécifiquement la détection des SPOOFS (Falsifiés)
POS_SPOOF_LABEL = 0

eval_accuracy = accuracy_score(all_targets, all_preds) * 100
eval_precision = (
    precision_score(
        all_targets, all_preds, pos_label=POS_SPOOF_LABEL, zero_division=0
    )
    * 100
)
eval_recall = (
    recall_score(
        all_targets, all_preds, pos_label=POS_SPOOF_LABEL, zero_division=0
    )
    * 100
)
eval_f1 = (
    f1_score(
        all_targets, all_preds, pos_label=POS_SPOOF_LABEL, zero_division=0
    )
    * 100
)

# Calcul de l'EER via votre fonction dédiée
eval_tracker = eval_epoch(model, eval_loader, device)
final_eval_eer = compute_eer(eval_tracker, eval_txt)

# 5. Affichage structuré du bilan complet
print("\n" + "=" * 50)
print("          BILAN D'ÉVALUATION COMPLET (EVAL)          ")
print("=" * 50)
print(f" ACCURACY  (Précision Globale)    : {eval_accuracy:.2f}%")
print(f" PRECISION (Détection Falsifiés) : {eval_precision:.2f}%")
print(f" RECALL    (Rappel / Sensibilité): {eval_recall:.2f}%")
print(f" F1-SCORE  (Score Harmonique)    : {eval_f1:.2f}%")
print(f" EER       (Equal Error Rate)    : {final_eval_eer:.2f}%")
print("=" * 50)

--- Chargement du meilleur modèle pour l'évaluation ---
Modèle chargé avec succès depuis : /content/drive/MyDrive/aasist/checkpoints/best_aasist_model.pth
Évaluation en cours sur le jeu Eval (71 237 audios)...

          BILAN D'ÉVALUATION COMPLET (EVAL)          
 ACCURACY  (Précision Globale)    : 95.26%
 PRECISION (Détection Falsifiés) : 99.71%
 RECALL    (Rappel / Sensibilité): 95.00%
 F1-SCORE  (Score Harmonique)    : 97.29%
 EER       (Equal Error Rate)    : 3.56%


Trying model.summary()


In [ ]:
print(model)


In [ ]:
!pip install torchinfo


In [ ]:
from torchinfo import summary

summary(model, input_size=(16, 64600))
